# ⚡ Coldframe Remotion Render on Google Colab → Direct Google Drive Delivery

This notebook renders the **Ako Ang Bagyo** 3D kinetic typography video (8,325 frames, 1080p @ 30 FPS) using the **Coldframe chunked architecture**:
1. **Single bundle pass**: compiles the Remotion project once.
2. **Parallel chunk renders**: renders frame segments as Matroska MKV + lossless PCM audio (`--codec=h264-mkv --audio-codec=pcm-16`).
3. **Zero re-encode lossless stitch**: joins all chunks instantaneously with FFmpeg concat.
4. **Direct Google Drive delivery**: writes directly to `/content/drive/MyDrive/Ako_Ang_Bagyo.mp4` so your render is preserved safely without download timeouts.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive'
print(f"Google Drive mounted successfully! Output will be delivered to: {OUT_DIR}/Ako_Ang_Bagyo.mp4")

## 2. Clone Repository & Setup Project

In [ ]:
!rm -rf /content/repo
!git clone https://github.com/edmer664/ako-ang-bagyo-remotion.git /content/repo
%cd /content/repo/my-video

## 3. Install Node.js 22 & Headless Graphics Libraries

In [ ]:
# Install Node.js 22 & ffmpeg
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get update -qq && sudo apt-get install -y -qq nodejs ffmpeg \
    libnss3 libdbus-1-3 libatk1.0-0t64 libatk-bridge2.0-0t64 libgbm1 libasound2t64 \
    libxrandr2 libxkbcommon0 libxfixes3 libxcomposite1 libxdamage1 libcups2t64 \
    libpango-1.0-0 libcairo2 fonts-noto-color-emoji || true

# Install project dependencies
!npm ci --no-audit --no-fund

## 4. Coldframe Bundle

In [ ]:
%cd /content/repo/my-video
# Bundle once to maximize rendering speed across all chunks
!npx remotion bundle src/index.ts --out-dir=coldframe-bundle
!npx remotion compositions coldframe-bundle

## 5. Coldframe Chunk Render

In [ ]:
import subprocess
import math
import os

TOTAL_FRAMES = 8325
CHUNKS = 8
COMPOSITION = "AkoAngBagyo"
CHUNKS_DIR = "/content/repo/my-video/coldframe-parts"
os.makedirs(CHUNKS_DIR, exist_ok=True)

frames_per_chunk = math.ceil(TOTAL_FRAMES / CHUNKS)

for i in range(CHUNKS):
    start = i * frames_per_chunk
    end = min((i + 1) * frames_per_chunk - 1, TOTAL_FRAMES - 1)
    chunk_file = f"{CHUNKS_DIR}/chunk-{i:02d}.mkv"
    
    if os.path.exists(chunk_file) and os.path.getsize(chunk_file) > 1000:
        print(f"✓ Chunk {i+1}/{CHUNKS} (frames {start}-{end}) already rendered. Skipping.")
        continue
        
    print(f"\n🚀 Rendering Chunk {i+1}/{CHUNKS}: Frames {start}-{end}...")
    cmd = [
        "npx", "remotion", "render", "coldframe-bundle", COMPOSITION, chunk_file,
        "--codec=h264-mkv", "--audio-codec=pcm-16",
        f"--frames={start}-{end}",
        "--gl=angle", "--concurrency=100%", "--timeout=180000"
    ]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        print(f"Retrying with software angle gl...")
        cmd[cmd.index("--gl=angle")] = "--gl=swangle"
        subprocess.run(cmd, check=True)

print("\n🎉 All chunks rendered successfully!")

## 6. Coldframe Lossless Stitch & Deliver to Google Drive

In [ ]:
import subprocess
import glob

parts = sorted(glob.glob("/content/repo/my-video/coldframe-parts/chunk-*.mkv"))
list_path = "/content/repo/my-video/concat_list.txt"
with open(list_path, "w") as f:
    for p in parts:
        f.write(f"file '{p}'\n")

FINAL_OUT = "/content/drive/MyDrive/Ako_Ang_Bagyo.mp4"
print(f"Stitching {len(parts)} chunks without re-encoding to: {FINAL_OUT}...")

# 1. Concat video stream losslessly
# 2. Encode joined lossless PCM audio to high-bitrate AAC 320k
concat_cmd = [
    "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_path,
    "-c:v", "copy", "-c:a", "aac", "-b:a", "320k", "-movflags", "+faststart",
    FINAL_OUT
]
subprocess.run(concat_cmd, check=True)

# Verify frame count
probe_cmd = [
    "ffprobe", "-v", "error", "-select_streams", "v:0", "-count_packets",
    "-show_entries", "stream=nb_read_packets", "-of", "default=nw=1:nk=1", FINAL_OUT
]
frames = subprocess.check_output(probe_cmd).decode().strip()
print(f"\n✅ Delivery Complete! Total frames: {frames} (Expected: 8325)")
print(f"📁 Video successfully saved to Google Drive at: {FINAL_OUT}")

## 7. Preview Video in Notebook

In [ ]:
from IPython.display import HTML
from base64 import b64encode

print("Video rendered and ready on your Google Drive!")